In [13]:
import pandas as pd
import os
import nltk
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [14]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/utsavlakshkar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/utsavlakshkar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/utsavlakshkar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
# Absolute path to the project root
BASE_DIR = os.path.abspath("..")

# Data directories
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "processed")

# Create processed directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
validation = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))

# Merge all datasets
raw_data = pd.concat([train, validation, test], ignore_index=True)

#Drop Id
raw_data.drop(columns=["id"], errors="ignore", inplace=True)

print(raw_data.head())
print(f"Total samples: {len(raw_data)}")

                                             article  \
0  By . Associated Press . PUBLISHED: . 14:11 EST...   
1  (CNN) -- Ralph Mata was an internal affairs li...   
2  A drunk driver who killed a young woman in a h...   
3  (CNN) -- With a breezy sweep of his pen Presid...   
4  Fleetwood are the only team still to have a 10...   

                                          highlights  
0  Bishop John Folda, of North Dakota, is taking ...  
1  Criminal complaint: Cop used his role to help ...  
2  Craig Eccleston-Todd, 27, had drunk at least t...  
3  Nina dos Santos says Europe must be ready to a...  
4  Fleetwood top of League One after 2-0 win at S...  
Total samples: 311971


In [18]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 311971 entries, 0 to 311970
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   article     311971 non-null  str  
 1   highlights  311971 non-null  str  
dtypes: str(2)
memory usage: 4.8 MB


In [19]:
raw_data.isnull().sum()

article       0
highlights    0
dtype: int64

In [20]:
raw_data.isna().sum()

article       0
highlights    0
dtype: int64

In [21]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
def columns_pre_processing(text_column):
    """
    Cleans and tokenizes each document in a text column.
    """
    
    processed_documents = []
    for text in text_column:
        if pd.isna(text) or not str(text).strip():
            processed_documents.append([])
            continue

        tokens = [
            LEMMATIZER.lemmatize(word)
            for sentence in sent_tokenize(text)
            for word in word_tokenize(sentence.lower())
            if word.isalpha() and word not in STOP_WORDS
        ]
        processed_documents.append(tokens)
    return processed_documents
    
def pre_processing(df):    
    df["article_tokenized"] = columns_pre_processing(df["article"])
    df["highlights_tokenized"] = columns_pre_processing(df["highlights"])
    return df

df=pre_processing(raw_data)  

In [22]:
print(df)

                                                  article  \
0       By . Associated Press . PUBLISHED: . 14:11 EST...   
1       (CNN) -- Ralph Mata was an internal affairs li...   
2       A drunk driver who killed a young woman in a h...   
3       (CNN) -- With a breezy sweep of his pen Presid...   
4       Fleetwood are the only team still to have a 10...   
...                                                   ...   
311966  Our young Earth may have collided with a body ...   
311967  A man facing trial for helping his former love...   
311968  A dozen or more metal implements are arranged ...   
311969  Brook Lopez dominated twin brother Robin with ...   
311970  A Chinese hospital is being painstakingly move...   

                                               highlights  \
0       Bishop John Folda, of North Dakota, is taking ...   
1       Criminal complaint: Cop used his role to help ...   
2       Craig Eccleston-Todd, 27, had drunk at least t...   
3       Nina dos Santos

In [23]:
output_file = os.path.join(OUTPUT_DIR, "preprocessed_dataset.csv")
df.to_csv(output_file, index=False)
print(f"Saved successfully at: {output_file}")

df.to_pickle(
    os.path.join(OUTPUT_DIR, "preprocessed_dataset.pkl")
)

Saved successfully at: /Users/utsavlakshkar/Documents/GitHub/NLP_Assignment_2_Text_Summarization_Using_Transformers/data/processed/preprocessed_dataset.csv
